In [5]:
import pandas as pd
import os 

# inputs
only_pivot = 0

lookup_agreement = '../lookup/lookup_agreement.csv'
lookup_song = '../lookup/lookup_song.csv'
inputdirectory1 = '../Combined/prelim_combined'
inputdirectory2 = '../Combined'
outputdirectory = '../'
headerfilename = 'combined_header.xlsx'
pivotfilename =  'combined_pivot.xlsx'
outputfilename = 'Limp_Biskit_combined.csv'


In [6]:

def find_xls_files(directory):
    xls_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.csv'):
                xls_files.append(os.path.join(root, file))
    return xls_files 
    
def Import_and_combine_single_files(directory): 
    print("Reading files and combining them in one data frame")
    listoffiles = find_xls_files(directory)
    print(f"\nTotal number of 'csv' files found is {len(listoffiles)}")
    for xls_file in listoffiles:
        print(xls_file)
    data_frames = []
    for file in listoffiles:
        print(f"\nAccessing {file}")
        try:
            df = pd.read_csv(file, header=0,low_memory=False)
        except ValueError as e:
            print(f"Error reading {file}: {e}")
        except Exception as e:
            print(f"Unexpected error reading {file}: {e}")
        print(f"It has {df.shape[0]} rows and {len(df.columns)} columns.")
        print(f"Royalty: {df['Net_roy_earn'].sum()}. Units: {df['Units'].sum()}") 
        data_frames.append(df)    
    df_comb = pd.concat(data_frames, ignore_index=True)
    print(f"\nThe combined DataFrame df_1_4 has {df_comb.shape[0]} rows and {len(df_comb.columns)} columns.")
    print(f"Royalty: {df_comb['Net_roy_earn'].sum()}. Units: {df_comb['Units'].sum()}")
    return df_comb

def compute_receipts(row):
    if pd.isna(row['Receipts.Computed']):
        return row['Price'] * row['Units']
    return row['Receipts.Computed']

def convert_percentage(value):
    return float(value.strip('%')) / 100


In [7]:
df_agreement = pd.read_csv(lookup_agreement,low_memory=False)
df_song = pd.read_csv(lookup_song,low_memory=False)

if only_pivot == 1:
    filename = os.path.join(inputdirectory2, outputfilename)
    df = pd.read_csv(filename,low_memory=False)
else: 
    df = Import_and_combine_single_files(inputdirectory1)
    columns_to_convert = ['Pckg_rate', 'Tax_rate', 'Part %']
    df[columns_to_convert] = df[columns_to_convert].astype(str)
    df['Pckg_rate'] = df['Pckg_rate'].apply(convert_percentage)
    df['Tax_rate'] = df['Tax_rate'].apply(convert_percentage)
    df['Part %'] = df['Part %'].apply(convert_percentage)
    df['Receipts.Computed'] = df['Receipts $']
    df['Receipts.Computed'] = df.apply(compute_receipts, axis=1)
    df['Gross_roy_earn'] = df['Net_roy_earn']/(1-df['Tax_rate'])
    df['Sales Date'] = pd.to_datetime(df['Sales Date'], format='%m/%y')
    df['Sales Date'] = df['Sales Date'].apply(lambda x: x.replace(year=2000 + x.year % 100))
    df['Sales Date'] = df['Sales Date'].fillna('N/A')
    df['Agreement_original']= df['Filename']
    df['Agreement_original']= df['Agreement_original'].str.slice(7)
    print("Merging on agreement")
    df = pd.merge(df, df_agreement, on='Agreement_original', how='left')
    df = df.drop(columns=['Agreement_original'])
    df.iloc[:, :12] = df.iloc[:, :12].fillna("N/A")
    df.iloc[:, 14:17] = df.iloc[:, 14:17].fillna("N/A")
    #df['Vendor_no'] = df['Vendor_no'].str.replace(".0","")
    columns_to_clean = ['Contract', 'Config', 'DSP', 'Acct_no','Seq_no','Selection', 'Payee_no','Price Level']
    for column in columns_to_clean:
        df[column] = df[column].str.replace('="', '').str.replace('"', '')
    print("Merging on Selection")
    df = pd.merge(df, df_song, on='Selection', how='left')
    print(f"Receipts: {df['Receipts $'].sum()}. Receipts.Computed: {df['Receipts.Computed'].sum()}. Units: {df['Units'].sum()}")
    df.info()
    filename2 = os.path.join(outputdirectory, outputfilename)
    df.to_csv(filename2, index=False)


Reading files and combining them in one data frame

Total number of 'csv' files found is 2
../Combined/prelim_combined/Limp_Biskit_csv_combined.csv
../Combined/prelim_combined/Limp_Biskit_htm_combined.csv

Accessing ../Combined/prelim_combined/Limp_Biskit_csv_combined.csv
It has 1662007 rows and 28 columns.
Royalty: 2993789.169999999. Units: 30473990214.0

Accessing ../Combined/prelim_combined/Limp_Biskit_htm_combined.csv
It has 905514 rows and 28 columns.
Royalty: 3764245.7000000007. Units: 7226071079.0

The combined DataFrame df_1_4 has 2567521 rows and 28 columns.
Royalty: 6758034.869999998. Units: 37700061293.0
Merging on agreement
Merging on Selection
Receipts: 87663365.87999998. Receipts.Computed: 140696995.93000004. Units: 37700061293.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2567521 entries, 0 to 2567520
Data columns (total 32 columns):
 #   Column             Dtype  
---  ------             -----  
 0   RoySys             object 
 1   Acct_no            object 
 2   

In [8]:
df_filter = df

#df_filter = df[df['Contract'].str.contains('S-0')]
#df_filter = df[df['Contract'].isin(['S-0001', 'S-0002', 'S-0003', 'S-0004', 'S-0006', 'S-0007', 'S-0010', 'S-0012', 'S-0202'])]
#df_filter = df[df['Contract'].isin(['S-0001', 'S-0002', 'S-0003', 'S-0004', 'S-0006', 'S-0007', 'S-0010', 'S-0012', 'S-0202'])]

#df_filter = df[df['Statement Period'] == '2023-H2'].iloc[:, :] 
#df_filter = df_filter[df_filter['Account No.'] == 'S10002212'].iloc[:, :] 

print(f"The filtered dataframe has {df_filter.shape[0]} rows and {len(df_filter.columns)} columns.")

print("\nColumn names are:")
for column in df_filter.columns:
    print(column)

val1 = 'Receipts.Computed'
val2 = 'Units'
val3 = 'Gross_roy_earn'
val4 = 'Net_roy_earn'
val5 = 'Agreement'

val1crit = 'sum'
val2crit = 'sum'
val3crit = 'sum'
val4crit = 'sum'
val5crit = 'count'

x0 = 'Filename'
x1 = 'RoySys'
x2 = 'Acct_no'
x3 = 'Acct_Qtr'
x4 = 'Seq_no'
x5 = 'Payee_no'
x6 = 'Vendor_no'
x7 = 'Group_no'
x8 = 'Source'
x9 = 'Title'
x10 = 'DSP'
x11 = 'Sales Type'
x12 = 'SC'
x13 = 'Price Level'
x14 = 'Sales Date'
x15 = 'Selection'
x16 = 'Config'
x17 = 'Contract'
x18 = 'Pr_code'
x19 = 'Price'
x20 = 'Pckg_rate'
x21 = 'Roy_Rate'
x22 = 'Part %'
x23 = 'Eff_rate'
x25 = 'Receipts $'
x26 = 'Tax_rate'
x27 = 'Agreement'
x28 = 'SONG TITLE'

sortby = val2


pivot = pd.pivot_table(df_filter, 
                        values=[val1,val2,val4],
                        index=[x27], 
                        #columns= [x11],
                        fill_value='',
                        margins= False,
                        aggfunc={val1:val1crit, val2:val2crit, val4:val4crit})
filename3 = os.path.join(outputdirectory, pivotfilename)
pivot.to_excel(filename3, engine='openpyxl', index=True)
pivot


The filtered dataframe has 2567521 rows and 32 columns.

Column names are:
RoySys
Acct_no
Acct_Qtr
Seq_no
Payee_no
Vendor_no
Group_no
Source
Title
DSP
Sales Type
SC
Price Level
Sales Date
Selection
Config
Contract
Pr_code
Price
Pckg_rate
Roy_Rate
Part %
Eff_rate
Units
Receipts $
Tax_rate
Net_roy_earn
Filename
Receipts.Computed
Gross_roy_earn
Agreement
SONG TITLE


,Net_roy_earn,Receipts.Computed,Units
Agreement,,,
Limp Bizkit-LP4 Result '00 AGM,1047796.97,9.137456e+06,1.928143e+09
Limp Bizkit-LP5,209622.15,2.395415e+06,5.209574e+08
Family Values Tour- Limp Bizkit,56739.45,7.242547e+05,4.571133e+07
Limp Bizkit- EP- Geffen,401226.10,3.005158e+06,2.948323e+08
Limp Bizkit- L- Video,273653.87,8.025197e+05,3.034288e+08
Limp Bizkit- LP 1-3 96 AGMNT,4113195.29,1.230434e+08,3.392512e+10
Limp Bizkit- LP2 Video,370212.24,9.021137e+05,3.208413e+08
Limp Bizkit- LP3 Video,104109.77,3.207545e+05,1.363537e+08
Limp Bizkit- LP4 Video,151397.14,3.053276e+05,1.870777e+08
